# So sánh Tokenizer: Underthesea vs PyVi
Notebook này thực hiện word segmentation trên tập dữ liệu raw bằng hai thư viện phổ biến nhất cho tiếng Việt để đánh giá sự khác biệt.

In [1]:
import pandas as pd
from underthesea import word_tokenize as uts_tokenize
from pyvi import ViTokenizer as pyvi_tokenize
import py_vncorenlp
import random
from tqdm import tqdm

# Load data
df = pd.read_csv('../raw/clickbait_dataset_vietnamese.csv')
print(f"Tổng số mẫu: {len(df)}")

# Chọn cột title để test
texts = df['title'].fillna("").astype(str).tolist()

Tổng số mẫu: 3414


In [2]:
model_path = '/home/nato/vncorenlp_data'

In [3]:
rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=model_path)

2026-05-01 10:31:10 INFO  WordSegmenter:24 - Loading Word Segmentation model


In [6]:
def format_vncorenlp(res):
    return " ".join(res).strip()


results = []
print("Đang tiến hành tokenize...")

for text in tqdm(texts):
    # Underthesea
    uts_res = uts_tokenize(text, format="text")
    
    # PyVi
    pyvi_res = pyvi_tokenize.tokenize(text)
    
    # VNcoreNLP
    raw_vnNLP = rdrsegmenter.word_segment(text)
    vnNLP_res = format_vncorenlp(raw_vnNLP)

    results.append({
        'original': text,
        'underthesea': uts_res,
        'pyvi': pyvi_res,
        'VN_coreNLP': vnNLP_res,
        'is_different': len({uts_res, pyvi_res, vnNLP_res}) > 1
    })

res_df = pd.DataFrame(results)
diff_df = res_df[res_df['is_different'] == True]

print(f"\nSố lượng mẫu có kết quả khác nhau: {len(diff_df)} ({len(diff_df)/len(df)*100:.2f}%)")

Đang tiến hành tokenize...


100%|██████████| 3414/3414 [00:02<00:00, 1222.37it/s]


Số lượng mẫu có kết quả khác nhau: 2205 (64.59%)


### Hiển thị ngẫu nhiên 5 mẫu có sự khác biệt

In [9]:
if len(diff_df) > 0:
    print("Hiển thị ngẫu nhiên 5 mẫu có sự khác biệt:\n")
    samples = diff_df.sample(min(5, len(diff_df)))
    for i, row in samples.iterrows():
        print("=" * 80)
        print(f"Original   : {row['original']}")
        print(f"Underthesea: {row['underthesea']}")
        print(f"PyVi       : {row['pyvi']}")
        print(f"VN_coreNLP : {row['VN_coreNLP']}")
else:
    print("Không có sự khác biệt nào giữa hai tokenizer trên tập dữ liệu này.")

Hiển thị ngẫu nhiên 5 mẫu có sự khác biệt:

Original   : CLB Đà Nẵng đề nghị đổi sân đá trận play-off - Báo Sài Gòn Giải Phóng
Underthesea: CLB Đà_Nẵng đề_nghị đổi sân đá trận play-off - Báo Sài_Gòn Giải_Phóng
PyVi       : CLB Đà_Nẵng đề_nghị đổi sân đá trận play - off - Báo Sài_Gòn Giải_Phóng
VN_coreNLP : CLB Đà_Nẵng đề_nghị đổi sân đá trận play-off - Báo Sài_Gòn Giải_Phóng
Original   : Đăng ký 'trại hè quân đội' giả trên mạng, một phụ huynh bị lừa 2,7 tỉ đồng
Underthesea: Đăng_ký ' trại hè quân_đội ' giả trên mạng , một phụ_huynh bị lừa 2,7 tỉ đồng
PyVi       : Đăng_ký ' trại_hè quân_đội ' giả trên mạng , một phụ_huynh bị lừa 2,7 tỉ đồng
VN_coreNLP : Đăng_ký ' trại_hè quân_đội ' giả trên mạng , một phụ_huynh bị lừa 2,7 tỉ đồng
Original   : Người tuổi này nên chọn nhà hướng Đông Tứ Trạch giúp thu hút tài lộc, gia đạo thuận hòa và sức khỏe dồi dào - Chuyên trang Gia Đình & Xã Hội - Báo Sức khỏe & Đời sống
Underthesea: Người tuổi này nên chọn nhà hướng Đông_Tứ_Trạch giúp thu_hút tài_lộc